# Workshop — RAG sobre papers académicos con LangChain y NVIDIA NIM

En este workshop construirás un sistema RAG completo alimentado con dos papers fundamentales:

- **Attention Is All You Need** — Vaswani et al. (2017)
- **Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks** — Lewis et al. (2020)

El notebook descarga los PDFs desde arXiv, extrae su texto, crea embeddings y responde preguntas usando únicamente el contenido indexado.

**Stack:** MarkItDown · LangChain · NVIDIA NIM · Qdrant

| Parte | Tarea |
|---|---|
| 0 | Instalar y configurar Colab |
| 1 | Descargar y convertir PDFs con MarkItDown |
| 2 | Crear `Document` y aplicar chunking |
| 3 | Generar embeddings NVIDIA e indexar en Qdrant |
| 4 | Construir el pipeline RAG con LCEL |
| 5 | Evaluar mediante preguntas con *ground truth* |

**Modelos:**
- Embeddings: `nvidia/nv-embedqa-e5-v5`
- Generación: `nvidia/nemotron-3.5-super-vl-120b-a12b`


> **Entrega · Daniel Muñeton (SI3003).** Copia con el Bloque B y varios retos aplicados:
> - **B1** · quiz con ground truth (celdas originales de la Parte 5).
> - **B2** · experimentos con `chunk_size`, `k` y `score_threshold`.
> - **B3** · preguntas trampa (¿admite que no sabe?).
> - **B4** · preguntas en espanol sobre papers en ingles.
> - **B5** · mi propio quiz con ground truth.
> - **C2** · tercer paper (HNSW) anadido al diccionario `PAPERS`.
> - **C3** · app Gradio con filtro por paper.
> - **C5** · MMR vs similitud pura.
> - **C6** · persistencia del indice en disco.

In [ ]:
# Dependencias para Google Colab
%pip install -qU "markitdown[pdf]" langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv requests gradio


In [ ]:
import os
import getpass
from pathlib import Path
from typing import List

import requests
from dotenv import load_dotenv

load_dotenv()

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
GENERATION_MODEL = "nvidia/nemotron-3.5-super-vl-120b-a12b"
EMBEDDING_MODEL = "nvidia/nv-embedqa-e5-v5"

print("✅ Configuración NVIDIA NIM lista")

# --- Parche de red: evita ReadTimeout (read timeout=60) en endpoints NIM lentos ---
# La libreria usa requests con timeout fijo de 60s; aqui elevamos el timeout
# por defecto a 300s para todas las peticiones que no especifiquen uno.
import requests.adapters as _ra
if not getattr(_ra.HTTPAdapter, "_timeout_patched", False):
    _orig_send = _ra.HTTPAdapter.send
    def _send(self, request, **kw):
        if kw.get("timeout") is None:
            kw["timeout"] = 300
        return _orig_send(self, request, **kw)
    _ra.HTTPAdapter.send = _send
    _ra.HTTPAdapter._timeout_patched = True
    print("\u2705 Timeout de red ampliado a 300s")


---
## Parte 1 — Descargar y convertir los PDFs

La celda descarga los artículos únicamente si todavía no existen en `papers/`. Después, MarkItDown convierte cada PDF a texto conservando buena parte de su estructura.

> Si una sesión de Colab se reinicia, los archivos temporales se pierden y se descargarán nuevamente.


> **C2.** Agrego un tercer paper al diccionario `PAPERS`: *HNSW* (Malkov & Yashunin, 2016, arXiv 1603.09320). Se descarga, convierte e indexa igual que los otros dos, de modo que el RAG tambien responde preguntas sobre busqueda aproximada por grafos.

In [ ]:
from markitdown import MarkItDown

DOCS_DIR = Path("papers")
DOCS_DIR.mkdir(parents=True, exist_ok=True)

PAPERS = {
    "attention_paper.pdf": "https://arxiv.org/pdf/1706.03762",
    "rag_paper.pdf": "https://arxiv.org/pdf/2005.11401",
    # C2 · tercer paper: HNSW (Malkov & Yashunin, 2016)
    "hnsw_paper.pdf": "https://arxiv.org/pdf/1603.09320",
}


def download_file(url: str, destination: Path) -> None:
    if destination.exists() and destination.stat().st_size > 10_000:
        print(f"↪ Ya existe: {destination.name}")
        return

    response = requests.get(url, timeout=90)
    response.raise_for_status()
    if not response.content.startswith(b"%PDF"):
        raise ValueError(f"La descarga de {url} no produjo un PDF válido")
    destination.write_bytes(response.content)
    print(f"⬇ Descargado: {destination.name} ({destination.stat().st_size / 1e6:.1f} MB)")


for filename, url in PAPERS.items():
    download_file(url, DOCS_DIR / filename)

md_converter = MarkItDown(enable_plugins=False)


def load_pdf(path: str | Path) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Archivo no encontrado: {path}")
    result = md_converter.convert(str(path))
    text = result.text_content
    if not text or not text.strip():
        raise ValueError(f"No se pudo extraer texto de {path.name}")
    return text


attention_text = load_pdf(DOCS_DIR / "attention_paper.pdf")
rag_text = load_pdf(DOCS_DIR / "rag_paper.pdf")
hnsw_text = load_pdf(DOCS_DIR / "hnsw_paper.pdf")  # C2

print(f"✅ attention_paper.pdf → {len(attention_text):,} caracteres")
print(f"✅ rag_paper.pdf       → {len(rag_text):,} caracteres")
print(f"✅ hnsw_paper.pdf      → {len(hnsw_text):,} caracteres")  # C2
print(f"\nInicio del paper de Attention:\n{attention_text[:300]}...")


---
## Parte 2 — Convertir a `Document` y aplicar chunking

LangChain utiliza objetos `Document` con dos campos principales:

- `page_content`: contenido textual del fragmento.
- `metadata`: título, archivo de origen e índice del fragmento.

Usaremos fragmentos de 900 caracteres con solapamiento de 120. Es un punto de partida didáctico; no es un valor universal.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


def pdf_to_documents(text: str, title: str, source: str) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=900,
        chunk_overlap=120,
        separators=["\n\n", "\n", ".", " "],
    )
    documents = []
    for index, chunk in enumerate(splitter.split_text(text)):
        if len(chunk.strip()) < 30:
            continue
        documents.append(Document(
            page_content=chunk,
            metadata={"title": title, "source": source, "chunk_index": index},
        ))
    return documents


attention_docs = pdf_to_documents(
    attention_text,
    title="Attention Is All You Need",
    source="attention_paper.pdf",
)
rag_docs = pdf_to_documents(
    rag_text,
    title="Retrieval-Augmented Generation",
    source="rag_paper.pdf",
)
hnsw_docs = pdf_to_documents(  # C2
    hnsw_text,
    title="Efficient and robust approximate nearest neighbor search using HNSW",
    source="hnsw_paper.pdf",
)
all_docs = attention_docs + rag_docs + hnsw_docs

print(f"Attention → {len(attention_docs)} chunks")
print(f"RAG       → {len(rag_docs)} chunks")
print(f"HNSW      → {len(hnsw_docs)} chunks")  # C2
print(f"Total     → {len(all_docs)} chunks")
print("\nEjemplo de metadatos:", all_docs[0].metadata)


---
## Parte 3 — Embeddings NVIDIA e índice Qdrant

`NVIDIAEmbeddings` distingue automáticamente entre embeddings de documentos y consultas mediante `embed_documents()` y `embed_query()`. La dimensión se obtiene directamente del modelo para evitar valores codificados manualmente.

Qdrant se ejecuta en memoria; al reiniciar el runtime se reconstruye el índice.


In [ ]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from langchain_qdrant import QdrantVectorStore
import time

COLLECTION = "academic_papers_nvidia"

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate="END")


def embed_con_reintentos(textos, intentos=4, espera=5):
    """Embebe texto por texto (peticiones pequenas) con reintentos. Evita el
    ReadTimeout de una unica peticion gigante en add_documents()."""
    vectores = []
    for i, t in enumerate(textos):
        for intento in range(intentos):
            try:
                vectores.append(embeddings.embed_query(t))
                break
            except Exception as e:
                if intento == intentos - 1:
                    raise
                print(f"  reintentando chunk {i}/{len(textos)} (intento {intento+1}): {type(e).__name__}")
                time.sleep(espera)
        if (i + 1) % 25 == 0:
            print(f"  ... {i+1}/{len(textos)} chunks embebidos")
    return vectores


textos = [d.page_content for d in all_docs]
metadatas = [d.metadata for d in all_docs]
vectores = embed_con_reintentos(textos)
EMBEDDING_DIM = len(vectores[0])

qdrant_client = QdrantClient(":memory:")
qdrant_client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
)

vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION,
    embedding=embeddings,
)

# Insertamos los puntos (vector + payload) directamente con el cliente de
# Qdrant, usando el formato de payload que espera QdrantVectorStore.
puntos = [
    PointStruct(id=i, vector=v, payload={"page_content": t, "metadata": m})
    for i, (t, v, m) in enumerate(zip(textos, vectores, metadatas))
]
qdrant_client.upsert(collection_name=COLLECTION, points=puntos)

print(f"✅ {len(puntos)} chunks indexados en Qdrant")
print(f"✅ Dimensión de embeddings: {EMBEDDING_DIM}")

---
## Parte 4 — Pipeline RAG con LCEL

El pipeline conecta recuperación, construcción del prompt, generación con Nemotron y extracción del texto final. El prompt obliga al modelo a reconocer cuando el contexto recuperado no contiene la respuesta.


In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatNVIDIA(
    model=GENERATION_MODEL,
    temperature=0.2,
    top_p=0.95,
    max_completion_tokens=800,
)

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 5, "score_threshold": 0.25},
)


def format_docs(documents: List[Document]) -> str:
    return "\n\n---\n\n".join(
        f"[{doc.metadata.get('title', 'Untitled')} | {doc.metadata.get('source', '')}]"
        f"\n{doc.page_content}"
        for doc in documents
    )


prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Answer using only the supplied academic-paper excerpts. "
        "If the excerpts do not contain enough evidence, say so explicitly. "
        "Answer in the same language as the question and be concise. "
        "Mention the paper title supporting the answer.",
    ),
    ("human", "CONTEXT:\n{context}\n\nQUESTION:\n{question}"),
])

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

test_question = "What is the Transformer architecture?"
test_answer = rag_chain.invoke(test_question)
print("Pregunta:", test_question)
print("Respuesta:", test_answer)


---
## Parte 5 — Quiz sobre los papers

Ejecuta cada pregunta y compara la respuesta del RAG con el *ground truth*. Las preguntas están en inglés para coincidir con el idioma de los artículos; el prompt pide responder en el mismo idioma de la consulta.

Además de la corrección factual, observa si el contexto recuperado proviene del paper adecuado.


In [ ]:
# ── Pregunta 1 ────────────────────────────────────────────────
q1 = "What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?"

answer1 = rag_chain.invoke(q1)
print(f"Q: {q1}")
print(f"A: {answer1}")

<details>
<summary>✅ Ground Truth — Pregunta 1</summary>

El Transformer (big model) obtuvo un BLEU score de **28.4** en la tarea de
traducción WMT 2014 English-to-German, superando los mejores resultados
previos (incluyendo ensembles) por más de 2 puntos BLEU.
*(Fuente: Abstract y Section 6.1 — Attention Is All You Need)*
</details>

In [ ]:
# ── Pregunta 2 ────────────────────────────────────────────────
q2 = "How does RAG combine parametric and non-parametric memory?"

answer2 = rag_chain.invoke(q2)
print(f"Q: {q2}")
print(f"A: {answer2}")

<details>
<summary>✅ Ground Truth — Pregunta 2</summary>

RAG combina **memoria paramétrica** (un modelo seq2seq pre-entrenado, BART-large)
con **memoria no-paramétrica** (un índice denso de vectores de Wikipedia).
El retriever (DPR) recupera los top-K documentos relevantes para la query,
y el generador (BART) los usa como contexto adicional para producir la respuesta.
Ambos componentes se entrenan end-to-end de forma conjunta.
*(Fuente: Abstract y Section 2 — RAG paper)*
</details>

In [ ]:
# ── Pregunta 3 ────────────────────────────────────────────────
q3 = "What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?"

answer3 = rag_chain.invoke(q3)
print(f"Q: {q3}")
print(f"A: {answer3}")

<details>
<summary>✅ Ground Truth — Pregunta 3</summary>

El factor de escala **1/√dk** se aplica para contrarrestar el efecto de que,
con valores grandes de dk, los productos punto crecen en magnitud y empujan
la función softmax hacia regiones con gradientes muy pequeños.
Esto haría el entrenamiento inestable. La división por √dk mantiene los
productos punto en un rango con gradientes útiles.
*(Fuente: Section 3.2.1 — Attention Is All You Need)*
</details>

In [ ]:
# ── Pregunta 4 ────────────────────────────────────────────────
q4 = "What are the two RAG model variants proposed in the paper and how do they differ?"

answer4 = rag_chain.invoke(q4)
print(f"Q: {q4}")
print(f"A: {answer4}")

<details>
<summary>✅ Ground Truth — Pregunta 4</summary>

Los dos variantes son:
- **RAG-Sequence**: usa el **mismo documento recuperado** para generar toda
  la secuencia de salida. Trata el documento como una variable latente única.
- **RAG-Token**: puede usar un **documento diferente para cada token** generado,
  permitiendo combinar información de múltiples fuentes en una sola respuesta.

RAG-Token es más flexible pero más costoso computacionalmente.
*(Fuente: Section 2.1 — RAG paper)*
</details>

In [ ]:
# ── Pregunta 5 ────────────────────────────────────────────────
q5 = "How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?"

answer5 = rag_chain.invoke(q5)
print(f"Q: {q5}")
print(f"A: {answer5}")

<details>
<summary>✅ Ground Truth — Pregunta 5</summary>

El Transformer base usa **h = 8** attention heads en paralelo.
Para cada cabeza, las dimensiones son **dk = dv = dmodel/h = 512/8 = 64**.
El costo computacional total es similar al de single-head attention con
dimensionalidad completa, gracias a las dimensiones reducidas por cabeza.
*(Fuente: Section 3.2.2 — Attention Is All You Need)*
</details>

---
## Bonus — Pregunta entre ambos papers

Esta pregunta exige conectar información de los dos artículos. Además de generar la respuesta, inspeccionaremos los títulos y archivos de los fragmentos recuperados.


In [ ]:
q_bonus = ("The RAG paper uses FAISS for fast retrieval with an HNSW approximation. "
           "The Attention paper proposes the Transformer architecture. "
           "What seq2seq model does RAG use as its generator component, "
           "and is it based on the Transformer?")

answer_bonus = rag_chain.invoke(q_bonus)

# Mostrar también qué chunks recuperó
docs_retrieved = retriever.invoke(q_bonus)
sources = [(d.metadata["title"], d.metadata["source"]) for d in docs_retrieved]

print(f"Q: {q_bonus}\n")
print(f"A: {answer_bonus}\n")
print(f"Fuentes recuperadas:")
for title, source in sources:
    print(f"  [{title}] — {source}")

<details>
<summary>✅ Ground Truth — Bonus</summary>

RAG usa **BART-large** como generador (Section 2.3 del RAG paper).
BART es un modelo seq2seq pre-entrenado basado en la arquitectura **Transformer**
(referencia [58] en el RAG paper = el paper "Attention Is All You Need").
El RAG paper cita explícitamente: "BART-large, a pre-trained seq2seq transformer".

Esta pregunta demuestra el valor de RAG: el sistema conecta información de
dos documentos distintos para construir una respuesta completa.
</details>

---
## Reflexión final

1. **¿Qué ocurrió cuando el contexto no contenía información suficiente?** ¿El modelo reconoció la limitación o inventó una respuesta?
2. **¿Qué fragmentos recuperó para la pregunta bonus?** ¿Aparecieron ambos papers?
3. **¿Qué efecto tendría modificar `chunk_size`, `chunk_overlap`, `k` o `score_threshold`?**
4. **¿Por qué conviene separar el modelo de embeddings del modelo generativo?**
5. **¿Qué cambiarías para llevar este prototipo a producción?** Considera persistencia, control de acceso, evaluación, observabilidad y costos.


---
## B2 · Experimentos con `chunk_size`, `k` y `score_threshold`

Reconstruyo el indice para distintos `chunk_size` y mido como cambian las respuestas y las fuentes. Tambien varío `k` y `score_threshold` con el indice actual. Anota los resultados en `RESPUESTAS.md` (B2).

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

def construir_indice(docs, nombre_coleccion):
    dim = len(embeddings.embed_query("dimension probe"))
    client = QdrantClient(":memory:")
    client.create_collection(
        collection_name=nombre_coleccion,
        vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
    )
    vs = QdrantVectorStore(client=client, collection_name=nombre_coleccion, embedding=embeddings)
    vs.add_documents(docs)
    return vs

PREGUNTA_B2 = "How many attention heads does the base Transformer use?"

# 1) Efecto de chunk_size
for cs in (400, 900, 1500):
    docs_cs = (
        pdf_to_documents(attention_text, "Attention Is All You Need", "attention_paper.pdf")
        if False else []
    )
    docs_cs = []
    for txt, titulo, src in [
        (attention_text, "Attention Is All You Need", "attention_paper.pdf"),
        (rag_text, "Retrieval-Augmented Generation", "rag_paper.pdf"),
    ]:
        from langchain_text_splitters import RecursiveCharacterTextSplitter
        sp = RecursiveCharacterTextSplitter(chunk_size=cs, chunk_overlap=min(120, cs//8),
                                            separators=["\n\n", "\n", ".", " "])
        from langchain_core.documents import Document
        for idx, ch in enumerate(sp.split_text(txt)):
            if len(ch.strip()) >= 30:
                docs_cs.append(Document(page_content=ch, metadata={"title": titulo, "source": src, "chunk_index": idx}))
    vs_cs = construir_indice(docs_cs, f"b2_cs_{cs}")
    r = vs_cs.as_retriever(search_type="similarity", search_kwargs={"k": 3}).invoke(PREGUNTA_B2)
    print(f"chunk_size={cs:>4}  chunks={len(docs_cs):>4}  top sources={[d.metadata['source'] for d in r]}")

# 2) Efecto de k y score_threshold sobre el indice actual
print("\nVariando k y score_threshold (indice actual):")
for k, thr in [(3, 0.25), (5, 0.25), (5, 0.4), (8, 0.2)]:
    docs = vectorstore.as_retriever(search_type="similarity_score_threshold",
                                    search_kwargs={"k": k, "score_threshold": thr}).invoke(PREGUNTA_B2)
    print(f"k={k} thr={thr} -> {len(docs)} chunks, fuentes={[d.metadata['source'] for d in docs]}")

---
## B3 · Preguntas trampa: ¿el RAG admite que no sabe?

Preguntas cuya respuesta NO esta en los papers. El prompt de la Parte 4 obliga a reconocer la falta de evidencia.

In [ ]:
PREGUNTAS_TRAMPA = [
    "What is the training cost in US dollars of GPT-4?",
    "Who won the 2022 FIFA World Cup according to these papers?",
    "What learning rate did BERT use for fine-tuning on SQuAD?",
]
for p in PREGUNTAS_TRAMPA:
    print("Q:", p)
    print("A:", rag_chain.invoke(p).strip())
    print("-" * 70)

---
## B4 · Preguntas en espanol sobre papers en ingles

El prompt pide responder en el idioma de la pregunta: debe contestar en espanol citando el paper en ingles.

In [ ]:
PREGUNTAS_ES = [
    "¿Cuantas cabezas de atencion usa el Transformer base y que dimension tiene cada una?",
    "¿Que dos variantes del modelo RAG existen y en que se diferencian?",
    "¿Que mecanismo usa el Transformer para incorporar el orden de las palabras?",
]
for p in PREGUNTAS_ES:
    print("P:", p)
    print("R:", rag_chain.invoke(p).strip())
    print("-" * 70)

---
## B5 · Mi propio quiz con ground truth

Cinco preguntas nuevas escritas por mi, con su respuesta de referencia, mas una pregunta que cruza ambos papers mostrando las fuentes.

In [ ]:
MI_QUIZ = [
    ("What residual + normalization structure does each Transformer sub-layer use?",
     "LayerNorm(x + Sublayer(x)): conexion residual seguida de layer normalization (Attention)."),
    ("What are the parametric and non-parametric components of RAG?",
     "Parametrico: seq2seq BART-large. No parametrico: indice denso de Wikipedia accedido por DPR (RAG)."),
    ("How does RAG-Token differ from RAG-Sequence at generation time?",
     "RAG-Token puede usar un documento distinto por token; RAG-Sequence usa el mismo para toda la secuencia (RAG)."),
    ("How many encoder and decoder layers does the base Transformer have?",
     "N = 6 capas identicas en encoder y en decoder (Attention)."),
    ("What problem of large parametric language models does RAG aim to mitigate?",
     "Acceder y actualizar conocimiento factual y reducir alucinaciones, con memoria no parametrica actualizable (RAG)."),
]
for pregunta, gt in MI_QUIZ:
    print("Q:", pregunta)
    print("RAG:", rag_chain.invoke(pregunta).strip())
    print("Ground truth:", gt)
    print("-" * 70)

BONUS_B5 = ("Both papers rely on the Transformer in some way: which component of "
            "RAG is a Transformer-based model, and which paper introduced that architecture?")
print("### Bonus (cruza papers) ###")
print("A:", rag_chain.invoke(BONUS_B5).strip())
for d in retriever.invoke(BONUS_B5):
    print("  fuente:", d.metadata["title"], "|", d.metadata["source"])

---
## C3 · App Gradio para los papers con filtro por paper

Interfaz Gradio con un desplegable que filtra la recuperacion por `metadata.source` (usando un `Filter` de Qdrant), ademas de la opcion "Todos los papers".

In [ ]:
import gradio as gr
from qdrant_client import models as qmodels
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

OPCIONES = {
    "Todos los papers": None,
    "Attention Is All You Need": "attention_paper.pdf",
    "Retrieval-Augmented Generation": "rag_paper.pdf",
    "HNSW": "hnsw_paper.pdf",
}

def responder_filtrado(pregunta, paper):
    source = OPCIONES.get(paper)
    filtro = None
    if source is not None:
        filtro = qmodels.Filter(must=[qmodels.FieldCondition(
            key="metadata.source", match=qmodels.MatchValue(value=source))])
    r = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 5, "score_threshold": 0.25, "filter": filtro},
    )
    docs = r.invoke(pregunta)
    if not docs:
        return "Sin evidencia en el (los) paper(s) seleccionado(s).", ""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer using only the supplied excerpts. If not enough evidence, say so. "
                   "Answer in the question language and mention the paper title."),
        ("human", "CONTEXT:\n{context}\n\nQUESTION:\n{question}"),
    ])
    cadena = (lambda _: {"context": format_docs(docs), "question": pregunta}) | prompt | llm | StrOutputParser()
    respuesta = cadena.invoke(None)
    fuentes = "\n".join(f"- {d.metadata['title']} ({d.metadata['source']})" for d in docs)
    return respuesta, fuentes

with gr.Blocks(title="RAG de papers con filtro") as demo_papers:
    gr.Markdown("# RAG sobre papers · filtro por paper (C3)")
    q = gr.Textbox(label="Pregunta", lines=2)
    sel = gr.Dropdown(choices=list(OPCIONES.keys()), value="Todos los papers", label="Filtrar por paper")
    b = gr.Button("Preguntar", variant="primary")
    out_md = gr.Markdown()
    src_md = gr.Markdown()
    b.click(responder_filtrado, inputs=[q, sel], outputs=[out_md, src_md])

demo_papers.queue().launch(share=True, debug=True, show_error=True)

---
## C5 · MMR frente a similitud pura

Comparo las fuentes recuperadas por similitud pura vs MMR (que penaliza la redundancia). Observa si MMR trae chunks mas diversos.

In [ ]:
pregunta_c5 = "How do these papers combine neural networks with attention or retrieval?"

print("=== Similitud pura (k=5) ===")
r_sim = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})
for d in r_sim.invoke(pregunta_c5):
    print("  ", d.metadata["source"], "| chunk", d.metadata["chunk_index"], "|", d.page_content[:70].replace("\n", " "))

print("\n=== MMR (k=5, fetch_k=20) ===")
r_mmr = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 20})
for d in r_mmr.invoke(pregunta_c5):
    print("  ", d.metadata["source"], "| chunk", d.metadata["chunk_index"], "|", d.page_content[:70].replace("\n", " "))

---
## C6 · Persistencia del indice en disco

En vez de `QdrantClient(":memory:")`, uso `QdrantClient(path="./qdrant_papers")`. Si la coleccion existe, la reutilizo sin reindexar y mido el tiempo; si no, la creo. Tras reiniciar el runtime (ejecutando solo las Partes 0-2 para tener `all_docs`), esta celda debe responder sin reindexar.

In [ ]:
import time
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

PERSIST_PATH = "./qdrant_papers"
COL_DISK = "academic_papers_disk"

client_disk = QdrantClient(path=PERSIST_PATH)
existe = client_disk.collection_exists(COL_DISK)
print("¿Ya existe la coleccion en disco?", existe)

t0 = time.time()
if not existe:
    dim = len(embeddings.embed_query("dimension probe"))
    client_disk.create_collection(collection_name=COL_DISK,
        vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
    vs_disk = QdrantVectorStore(client=client_disk, collection_name=COL_DISK, embedding=embeddings)
    vs_disk.add_documents(all_docs)
    print(f"Indexado desde cero en {time.time()-t0:.2f} s")
else:
    vs_disk = QdrantVectorStore(client=client_disk, collection_name=COL_DISK, embedding=embeddings)
    print(f"Reutilizada sin reindexar en {time.time()-t0:.2f} s")

retriever_disk = vs_disk.as_retriever(search_type="similarity_score_threshold",
                                      search_kwargs={"k": 5, "score_threshold": 0.25})
docs = retriever_disk.invoke("What is scaled dot-product attention?")
print("Chunks recuperados desde disco:", len(docs))